# ❤️ Heart Disease Data Analytics Project
**Author:** Delsiya  
**Dataset:** Heart Disease Patient Dataset (1000 records)  
**Goal:** Descriptive statistics, EDA, feature engineering, correlation/hypothesis testing, predictive modeling, and visualization.

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

## 📂 2. Load Dataset

In [ ]:
df_raw = pd.read_csv('Dataset.csv', encoding='utf-8-sig')

# Drop completely empty columns (trailing commas in CSV)
df_raw = df_raw.dropna(axis=1, how='all')

# Rename columns for convenience
df_raw.columns = [
    'patient_id', 'age', 'gender', 'cholesterol', 'bmi',
    'heart_rate', 'glucose', 'systolic_bp', 'diastolic_bp', 'ecg_result'
]

print(f'Shape: {df_raw.shape}')
df_raw.head(10)

## 🧹 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Drop fully empty rows
df.dropna(how='all', inplace=True)

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# Check missing values
print('\nMissing values per column:')
print(df.isnull().sum())

# Fill missing numeric values with median
num_cols = ['age', 'cholesterol', 'bmi', 'heart_rate', 'glucose', 'systolic_bp', 'diastolic_bp']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

# Fill missing categorical with mode
for col in ['gender', 'ecg_result']:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

df.reset_index(drop=True, inplace=True)
print(f'\nClean dataset shape: {df.shape}')
df.dtypes

## 📊 4. Descriptive Statistics

In [ ]:
print('=== Overall Descriptive Statistics ===')
df[num_cols].describe().round(2)

In [ ]:
print('=== Gender Distribution ===')
print(df['gender'].value_counts())
print('\n=== ECG Result Distribution ===')
print(df['ecg_result'].value_counts())

In [ ]:
print('=== Descriptive Statistics by Gender ===')
df.groupby('gender')[num_cols].mean().round(2)

## 🔧 5. Feature Engineering

In [ ]:
# Cholesterol-to-age ratio
df['chol_age_ratio'] = (df['cholesterol'] / df['age']).round(3)

# BMI categories (WHO)
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_category'] = df['bmi'].apply(bmi_category)

# Age groups
df['age_group'] = pd.cut(
    df['age'], bins=[0, 30, 45, 60, 75, 100],
    labels=['<30', '30-45', '45-60', '60-75', '75+']
)

# Pulse pressure
df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']

# Abnormal ECG flag (binary target)
normal_ecg = ['Normal sinus rhythm']
df['ecg_abnormal'] = df['ecg_result'].apply(lambda x: 0 if x in normal_ecg else 1)

# High-risk flag: age>60 AND cholesterol>240 AND abnormal ECG
df['high_risk'] = ((df['age'] > 60) & (df['cholesterol'] > 240) & (df['ecg_abnormal'] == 1)).astype(int)

# Encode gender
df['gender_enc'] = LabelEncoder().fit_transform(df['gender'])  # Female=0, Male=1

print('Feature engineering complete.')
df[['chol_age_ratio', 'bmi_category', 'age_group', 'pulse_pressure', 'ecg_abnormal', 'high_risk']].head()

## 📈 6. Exploratory Data Analysis (EDA)

In [ ]:
# ---- Histogram: Cholesterol Distribution ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['cholesterol'], bins=30, color='steelblue', edgecolor='white')
ax.axvline(df['cholesterol'].mean(), color='red', linestyle='--', label=f"Mean: {df['cholesterol'].mean():.1f}")
ax.set_title('Cholesterol Distribution', fontsize=14)
ax.set_xlabel('Cholesterol (mg/dl)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('cholesterol_hist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Age Distribution by Gender ----
fig, ax = plt.subplots(figsize=(8, 4))
for gender, grp in df.groupby('gender'):
    ax.hist(grp['age'], bins=20, alpha=0.6, label=gender)
ax.set_title('Age Distribution by Gender', fontsize=14)
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('age_gender_hist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Pie Chart: ECG Result Categories ----
ecg_counts = df['ecg_result'].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    ecg_counts, labels=ecg_counts.index, autopct='%1.1f%%',
    startangle=140, pctdistance=0.82
)
ax.set_title('ECG Result Categories', fontsize=14)
plt.tight_layout()
plt.savefig('ecg_pie.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Scatter: BMI vs Systolic BP ----
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    df['bmi'], df['systolic_bp'],
    c=df['ecg_abnormal'], cmap='RdYlGn_r', alpha=0.6, edgecolors='none'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Abnormal ECG (1=Yes)')
ax.set_title('BMI vs Systolic Blood Pressure', fontsize=14)
ax.set_xlabel('BMI')
ax.set_ylabel('Systolic BP (mm Hg)')
# Trend line
m, b = np.polyfit(df['bmi'], df['systolic_bp'], 1)
ax.plot(sorted(df['bmi']), [m*x+b for x in sorted(df['bmi'])], 'r--', linewidth=1.5, label='Trend')
ax.legend()
plt.tight_layout()
plt.savefig('bmi_sysbp_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Heatmap: Correlation Matrix ----
corr_cols = ['age', 'cholesterol', 'bmi', 'heart_rate', 'glucose',
             'systolic_bp', 'diastolic_bp', 'pulse_pressure', 'ecg_abnormal']
corr = df[corr_cols].corr().round(2)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax, vmin=-1, vmax=1
)
ax.set_title('Correlation Matrix — Heart Disease Features', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- ECG Abnormal by Age Group ----
age_ecg = df.groupby('age_group')['ecg_abnormal'].mean() * 100
fig, ax = plt.subplots(figsize=(7, 4))
age_ecg.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Abnormal ECG Rate by Age Group (%)', fontsize=14)
ax.set_xlabel('Age Group')
ax.set_ylabel('Abnormal ECG (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('ecg_abnormal_by_age.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Boxplot: Cholesterol by ECG Result ----
fig, ax = plt.subplots(figsize=(10, 5))
order = df['ecg_result'].value_counts().index
sns.boxplot(data=df, x='ecg_result', y='cholesterol', order=order, ax=ax, palette='Set2')
ax.set_title('Cholesterol by ECG Result', fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('cholesterol_by_ecg.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Risk Stratification: High-Risk Groups ----
print('=== High-Risk Patients (Age>60, Cholesterol>240, Abnormal ECG) ===')
high_risk_df = df[df['high_risk'] == 1]
print(f'High-risk patients: {len(high_risk_df)} ({len(high_risk_df)/len(df)*100:.1f}%)')
print('\nHigh-risk breakdown by ECG result:')
print(high_risk_df['ecg_result'].value_counts())

# BMI category vs ECG abnormal
bmi_ecg = df.groupby('bmi_category')['ecg_abnormal'].mean() * 100
fig, ax = plt.subplots(figsize=(6, 4))
bmi_ecg.sort_values().plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Abnormal ECG Rate by BMI Category (%)', fontsize=14)
ax.set_xlabel('Abnormal ECG (%)')
plt.tight_layout()
plt.savefig('ecg_by_bmi.png', bbox_inches='tight')
plt.show()

## 🔬 7. Correlation and Hypothesis Testing

In [ ]:
# ---- Pearson Correlation: Cholesterol vs ECG Abnormal ----
r, p = stats.pearsonr(df['cholesterol'], df['ecg_abnormal'])
print(f'Pearson r(cholesterol, ecg_abnormal) = {r:.4f}, p = {p:.4f}')

# ---- Pearson Correlation: BMI vs Systolic BP ----
r2, p2 = stats.pearsonr(df['bmi'], df['systolic_bp'])
print(f'Pearson r(BMI, systolic_bp) = {r2:.4f}, p = {p2:.4f}')

# ---- Chi-Square: Gender vs ECG Abnormal ----
ct = pd.crosstab(df['gender'], df['ecg_abnormal'])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print(f'\nChi-Square (gender vs ecg_abnormal): chi2={chi2:.4f}, p={p_chi:.4f}, dof={dof}')

# ---- ANOVA: Cholesterol across ECG groups ----
ecg_groups = [grp['cholesterol'].values for _, grp in df.groupby('ecg_result')]
f_stat, p_anova = stats.f_oneway(*ecg_groups)
print(f'\nANOVA (cholesterol across ECG groups): F={f_stat:.4f}, p={p_anova:.4f}')

## 🤖 8. Predictive Modeling

In [ ]:
# ---- Prepare features ----
FEATURES = ['age', 'gender_enc', 'cholesterol', 'bmi', 'heart_rate',
            'glucose', 'systolic_bp', 'diastolic_bp', 'pulse_pressure', 'chol_age_ratio']
TARGET = 'ecg_abnormal'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')
print(f'Target class distribution (full): {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# ---- Helper: evaluate a model ----
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_te, y_pred)
    roc = roc_auc_score(y_te, y_proba) if y_proba is not None else None
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()

    print(f'\n{'='*50}')
    print(f'Model: {name}')
    print(f'  Test Accuracy  : {acc:.4f}')
    print(f'  ROC-AUC        : {roc:.4f}' if roc else '  ROC-AUC: N/A')
    print(f'  CV Accuracy    : {cv:.4f}')
    print(classification_report(y_te, y_pred, target_names=['Normal', 'Abnormal']))
    return model, acc, roc

# ---- Models ----
lr  = LogisticRegression(max_iter=1000, random_state=42)
dt  = DecisionTreeClassifier(max_depth=6, random_state=42)
rf  = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
gb  = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)

results = {}
lr_fit, acc_lr, roc_lr = evaluate_model('Logistic Regression', lr,  X_train_sc, X_test_sc, y_train, y_test)
dt_fit, acc_dt, roc_dt = evaluate_model('Decision Tree',        dt,  X_train,    X_test,    y_train, y_test)
rf_fit, acc_rf, roc_rf = evaluate_model('Random Forest',        rf,  X_train,    X_test,    y_train, y_test)
gb_fit, acc_gb, roc_gb = evaluate_model('Gradient Boosting',    gb,  X_train,    X_test,    y_train, y_test)

In [ ]:
# ---- Model Comparison Bar Chart ----
model_names = ['Logistic\nRegression', 'Decision\nTree', 'Random\nForest', 'Gradient\nBoosting']
accs  = [acc_lr, acc_dt, acc_rf, acc_gb]
aucs  = [roc_lr, roc_dt, roc_rf, roc_gb]

x = np.arange(len(model_names))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, aucs, width, label='ROC-AUC',  color='coral')
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — Accuracy & ROC-AUC', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Confusion Matrix: Best model (Random Forest) ----
y_pred_rf = rf_fit.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Abnormal']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix_rf.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- ROC Curve ----
fig, ax = plt.subplots(figsize=(7, 5))
for name, model, X_te, label_color in [
    ('Logistic Regression', lr_fit, X_test_sc, 'blue'),
    ('Decision Tree',       dt_fit, X_test,    'green'),
    ('Random Forest',       rf_fit, X_test,    'red'),
    ('Gradient Boosting',   gb_fit, X_test,    'orange'),
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_te)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_te)[:, 1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=label_color)

ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Feature Importance — Random Forest ----
importances = rf_fit.feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
feat_imp.plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=14)
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 Most Important Features:')
print(feat_imp.sort_values(ascending=False).head())

In [ ]:
# ---- Decision Tree Visualization (simplified) ----
dt_small = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_small.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(
    dt_small, feature_names=FEATURES, class_names=['Normal', 'Abnormal'],
    filled=True, rounded=True, ax=ax, fontsize=9
)
plt.title('Decision Tree (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.savefig('decision_tree.png', bbox_inches='tight')
plt.show()

## 📋 9. Summary & Key Findings

In [ ]:
print('='*60)
print('HEART DISEASE DATA ANALYTICS — SUMMARY')
print('='*60)
print(f"Total patients analysed : {len(df)}")
print(f"Abnormal ECG patients   : {df['ecg_abnormal'].sum()} ({df['ecg_abnormal'].mean()*100:.1f}%)")
print(f"High-risk patients      : {df['high_risk'].sum()} ({df['high_risk'].mean()*100:.1f}%)")
print(f"Avg cholesterol         : {df['cholesterol'].mean():.1f} mg/dl")
print(f"Avg BMI                 : {df['bmi'].mean():.1f}")
print(f"Avg glucose             : {df['glucose'].mean():.1f} mg/dl")
print(f"Avg heart rate          : {df['heart_rate'].mean():.1f} bpm")
print()
print('Best Model: Random Forest')
print(f'  Accuracy : {acc_rf:.4f}')
print(f'  ROC-AUC  : {roc_rf:.4f}')
print()
print('Key Predictors (Random Forest):')
for feat, imp in feat_imp.sort_values(ascending=False).head(5).items():
    print(f'  {feat:25s}: {imp:.4f}')

---
## 🌐 10. Streamlit App
The companion Streamlit app is in `app.py`. To launch it, run:
```bash
streamlit run app.py
```